# Self-Evolving AI Agents: A Complete Guide

This notebook demonstrates the concept of self-evolving agents that improve through feedback loops.

## What are Self-Evolving Agents?

Self-evolving agents are AI systems that:
1. Execute tasks with current configuration
2. Evaluate their own performance
3. Generate improved strategies based on feedback
4. Continuously adapt without human intervention

In [ ]:
# Install required packages
!pip install openai python-dotenv graphviz matplotlib pandas seaborn -q

In [ ]:
import os
from openai import OpenAI
from typing import List, Dict, Any
from datetime import datetime
import json
import re
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Initialize OpenAI client
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

print("✓ Environment loaded successfully")

## 1. Prompt Version Manager

Tracks all versions of prompts with metadata and timestamps.

In [ ]:
class PromptVersion:
    def __init__(self, version: int, prompt: str, model: str, metadata: dict = None):
        self.version = version
        self.prompt = prompt
        self.model = model
        self.timestamp = datetime.utcnow()
        self.metadata = metadata or {}

class PromptManager:
    def __init__(self, initial_prompt: str, model: str = "gpt-4o-mini"):
        self.versions = []
        self.versions.append(PromptVersion(0, initial_prompt, model, {"type": "initial"}))
    
    def get_current(self):
        return self.versions[-1]
    
    def update(self, new_prompt: str, metadata: dict = None):
        current = self.get_current()
        next_version = current.version + 1
        self.versions.append(PromptVersion(next_version, new_prompt, current.model, metadata))
        return self.versions[-1]
    
    def get_history(self):
        return [(v.version, v.prompt[:100] + "...", v.metadata) for v in self.versions]

print("✓ PromptManager class defined")

## 2. Multi-Criteria Evaluator

Evaluates agent output across multiple dimensions:
- **Relevance**: How well does output address the input?
- **Quality**: Coherence, grammar, presentation
- **Completeness**: Are all aspects addressed?
- **Length**: Appropriate length for the task

In [ ]:
class Evaluator:
    def __init__(self, client, pass_threshold: float = 0.75):
        self.client = client
        self.pass_threshold = pass_threshold
    
    def evaluate_relevance(self, input_text: str, output: str, task: str):
        prompt = f"""Evaluate the relevance of the output to the given input and task.

Task: {task}
Input: {input_text[:500]}
Output: {output[:500]}

Rate relevance from 0.0 to 1.0:
SCORE: [0.0-1.0]
REASONING: [Brief explanation]"""

        response = self.client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.3
        )
        return self._parse_eval(response.choices[0].message.content)
    
    def evaluate_quality(self, output: str, task: str):
        prompt = f"""Evaluate the quality of this output.

Task: {task}
Output: {output}

Rate quality from 0.0 to 1.0 based on coherence, grammar, and presentation:
SCORE: [0.0-1.0]
REASONING: [Brief explanation]"""

        response = self.client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.3
        )
        return self._parse_eval(response.choices[0].message.content)
    
    def evaluate_length(self, output: str):
        word_count = len(output.split())
        if 50 <= word_count <= 500:
            score = 1.0
            reasoning = f"Length appropriate ({word_count} words)"
        elif word_count < 50:
            score = word_count / 50
            reasoning = f"Too short ({word_count} words)"
        else:
            score = max(0.0, 1.0 - (word_count - 500) / 500)
            reasoning = f"Too long ({word_count} words)"
        return {"score": score, "passed": score >= self.pass_threshold, "reasoning": reasoning}
    
    def _parse_eval(self, content: str):
        score_match = re.search(r'SCORE:\s*([0-9.]+)', content)
        reasoning_match = re.search(r'REASONING:\s*(.+)', content, re.DOTALL)
        score = float(score_match.group(1)) if score_match else 0.5
        reasoning = reasoning_match.group(1).strip() if reasoning_match else "No reasoning"
        return {"score": score, "passed": score >= self.pass_threshold, "reasoning": reasoning}
    
    def evaluate_all(self, input_text: str, output: str, task: str):
        results = {
            "relevance": self.evaluate_relevance(input_text, output, task),
            "quality": self.evaluate_quality(output, task),
            "length": self.evaluate_length(output)
        }
        avg_score = sum(r["score"] for r in results.values()) / len(results)
        all_passed = all(r["passed"] for r in results.values())
        return {"average_score": avg_score, "passed": all_passed, "details": results}

print("✓ Evaluator class defined")

## 3. Self-Evolving Agent

The core agent that implements the feedback loop.

In [ ]:
class SelfEvolvingAgent:
    def __init__(self, initial_prompt: str, model: str = "gpt-4o-mini"):
        self.client = client
        self.prompt_manager = PromptManager(initial_prompt, model)
        self.evaluator = Evaluator(client)
        self.history = []
    
    def execute_task(self, task_description: str, input_text: str):
        current_prompt = self.prompt_manager.get_current()
        messages = [
            {"role": "system", "content": current_prompt.prompt},
            {"role": "user", "content": f"{task_description}\n\nInput:\n{input_text}"}
        ]
        response = self.client.chat.completions.create(
            model=current_prompt.model,
            messages=messages,
            temperature=0.7
        )
        return response.choices[0].message.content
    
    def improve_prompt(self, current_prompt: str, task: str, input_text: str, 
                      output: str, evaluation: dict):
        feedback = "\n".join([
            f"- {k}: {v['score']:.2f} ({'PASS' if v['passed'] else 'FAIL'}) - {v['reasoning']}"
            for k, v in evaluation['details'].items()
        ])
        
        meta_prompt = f"""You are a prompt optimization expert. Improve the system prompt based on feedback.

Current Prompt:
{current_prompt}

Task: {task}
Input: {input_text[:300]}...
Output: {output[:300]}...

Evaluation (Average: {evaluation['average_score']:.2f}):
{feedback}

Generate an improved system prompt that addresses the failing criteria.
Return ONLY the improved prompt, nothing else."""

        response = self.client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": meta_prompt}],
            temperature=0.3
        )
        return response.choices[0].message.content.strip()
    
    def evolve(self, task_description: str, input_text: str, 
               max_iterations: int = 3, target_score: float = 0.8):
        print(f"\n{'='*60}")
        print(f"Starting Self-Evolution Process")
        print(f"{'='*60}\n")
        
        for iteration in range(1, max_iterations + 1):
            print(f"\n--- Iteration {iteration}/{max_iterations} ---")
            
            # Execute task
            output = self.execute_task(task_description, input_text)
            print(f"Generated output ({len(output.split())} words)")
            
            # Evaluate
            evaluation = self.evaluator.evaluate_all(input_text, output, task_description)
            
            # Record history
            current_prompt = self.prompt_manager.get_current()
            self.history.append({
                "iteration": iteration,
                "output": output,
                "score": evaluation["average_score"],
                "passed": evaluation["passed"],
                "details": evaluation["details"],
                "prompt_version": current_prompt.version
            })
            
            # Print evaluation
            print(f"\nEvaluation Results:")
            print(f"  Average Score: {evaluation['average_score']:.3f}")
            print(f"  Status: {'✓ PASSED' if evaluation['passed'] else '✗ FAILED'}")
            for criterion, result in evaluation["details"].items():
                status = '✓' if result['passed'] else '✗'
                print(f"  {status} {criterion}: {result['score']:.3f}")
            
            # Check success
            if evaluation["passed"] and evaluation["average_score"] >= target_score:
                print(f"\n{'='*60}")
                print(f"✓ Target achieved! Final score: {evaluation['average_score']:.3f}")
                print(f"{'='*60}")
                break
            
            # Improve prompt if not last iteration
            if iteration < max_iterations:
                print(f"\n→ Improving prompt...")
                improved_prompt = self.improve_prompt(
                    current_prompt.prompt, task_description, input_text,
                    output, evaluation
                )
                self.prompt_manager.update(improved_prompt, {
                    "iteration": iteration,
                    "previous_score": evaluation["average_score"]
                })
                print(f"✓ Prompt updated to version {self.prompt_manager.get_current().version}")
        
        return self.history[-1]

print("✓ SelfEvolvingAgent class defined")

## 4. Example: Text Summarization with Self-Evolution

Let's see the agent improve itself through iterations.

In [ ]:
# Initialize agent with basic prompt
initial_prompt = "You are a helpful assistant that generates summaries."
agent = SelfEvolvingAgent(initial_prompt)

# Sample text to summarize
sample_text = """Artificial intelligence has made remarkable progress in recent years, 
particularly in the field of large language models. These models, trained on vast amounts 
of text data, can generate human-like text, answer questions, and perform various language 
tasks. However, traditional AI agents have a significant limitation: once deployed, they 
remain static. They don't learn from their mistakes or adapt to new situations without 
human intervention. This is where self-evolving agents come in. Self-evolving agents are 
AI systems that can monitor their own performance, identify weaknesses, and automatically 
improve their strategies. They implement a feedback loop: execute a task, evaluate the 
result, generate improvements, and try again. This approach combines the power of large 
language models with continuous learning, enabling agents to become more accurate and 
reliable over time without requiring constant human oversight."""

# Run self-evolution
result = agent.evolve(
    task_description="Create a concise summary of the following text",
    input_text=sample_text,
    max_iterations=3,
    target_score=0.80
)

print("\n\nFinal Output:")
print("="*60)
print(result["output"])
print("="*60)

## 5. Visualization: Evolution Progress

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Extract data
iterations = [h["iteration"] for h in agent.history]
scores = [h["score"] for h in agent.history]

# Create subplot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Score evolution
ax1.plot(iterations, scores, marker='o', linewidth=2, markersize=10, color='#2E86AB')
ax1.axhline(y=0.8, color='#A23B72', linestyle='--', label='Target Score')
ax1.set_xlabel('Iteration', fontsize=12)
ax1.set_ylabel('Average Score', fontsize=12)
ax1.set_title('Self-Evolution Progress', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_ylim(0, 1.05)

# Plot 2: Criteria scores
criteria_data = []
for h in agent.history:
    for criterion, result in h["details"].items():
        criteria_data.append({
            "Iteration": h["iteration"],
            "Criterion": criterion.capitalize(),
            "Score": result["score"]
        })

df = pd.DataFrame(criteria_data)
for criterion in df["Criterion"].unique():
    data = df[df["Criterion"] == criterion]
    ax2.plot(data["Iteration"], data["Score"], marker='o', label=criterion, linewidth=2)

ax2.set_xlabel('Iteration', fontsize=12)
ax2.set_ylabel('Score', fontsize=12)
ax2.set_title('Criteria Breakdown', fontsize=14, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_ylim(0, 1.05)

plt.tight_layout()
plt.show()

print("\nScore Improvement:")
print(f"Initial: {scores[0]:.3f}")
print(f"Final: {scores[-1]:.3f}")
print(f"Improvement: +{(scores[-1] - scores[0]):.3f}")

## 6. Prompt Evolution Visualization

Let's visualize how the prompts evolved.

In [ ]:
print("Prompt Evolution History:")
print("="*80)

for version, prompt, metadata in agent.prompt_manager.get_history():
    print(f"\nVersion {version}:")
    print(f"Metadata: {metadata}")
    print(f"Prompt: {prompt}")
    print("-"*80)

## 7. Advanced Example: Custom Evaluation Criteria

In [ ]:
# Try with different task
agent2 = SelfEvolvingAgent("You are an AI that writes professional emails.")

email_request = """Write a professional email to decline a job offer politely 
while expressing gratitude and keeping the door open for future opportunities."""

result = agent2.evolve(
    task_description=email_request,
    input_text="Company: Tech Solutions Inc. Position: Senior Developer",
    max_iterations=3,
    target_score=0.85
)

print("\n\nFinal Email:")
print("="*60)
print(result["output"])
print("="*60)

## 8. Summary and Key Takeaways

### What We Learned

1. **Self-Evolving Agents** continuously improve without human intervention
2. **Feedback Loops** are the core mechanism for improvement
3. **Multi-Criteria Evaluation** ensures holistic quality
4. **Meta-Prompting** uses AI to improve AI prompts
5. **Version Control** tracks all changes and enables rollback

### Applications

- Content generation that improves over time
- Customer service bots that adapt to user feedback
- Code generation agents that learn from errors
- Research assistants that refine search strategies
- Educational tutors that personalize approaches

### Next Steps

- Implement domain-specific evaluation criteria
- Add human-in-the-loop feedback
- Integrate with production systems
- Scale to multi-agent systems
- Add memory and context persistence